# gold layer — exploratory tests

Validates the five key properties of the gold layer using an **in-memory DuckDB**
populated with controlled fixtures. No file I/O, no warehouse side-effects.

Run cells top-to-bottom. Each section prints an assertion result and shows the
relevant query output.

In [1]:
import sys
sys.path.insert(0, '..')  # make src/ importable from tests/

import logging
import duckdb
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

print('DuckDB version:', duckdb.__version__)

DuckDB version: 1.5.4


## Fixture setup

The fixture covers three reference dates:

| Date | Day | Runs | Notes |
|---|---|---|---|
| 2024-01-08 | Mon | run 1 (superseded), run 2 (winner) | Two COMPLETED runs — winning-run test |
| 2024-01-14 | Sun | run 3 | Same ISO week as 2024-01-08 — closed-week test |
| 2024-01-15 | Mon | run 4 | Next ISO week — closed-week test |

Run 1 is older for 2024-01-08 (`started_at` 08:00 < 09:00) so run 2 wins.
Compliance must still see both.

In [2]:
conn = duckdb.connect(':memory:')

# ── Silver tables ────────────────────────────────────────────────────────────
conn.execute("""
    CREATE TABLE silver_reconciliation_runs (
        id BIGINT PRIMARY KEY, reference_date DATE NOT NULL, file_name VARCHAR,
        status VARCHAR NOT NULL, total_transactions INTEGER,
        started_at TIMESTAMPTZ, completed_at TIMESTAMPTZ,
        created_at TIMESTAMPTZ NOT NULL, source VARCHAR NOT NULL
    )
""")
conn.execute("""
    CREATE TABLE silver_reconciliation_results (
        id BIGINT PRIMARY KEY, run_id BIGINT NOT NULL,
        transaction_id VARCHAR NOT NULL, merchant_id VARCHAR,
        category VARCHAR NOT NULL,
        internal_amount DECIMAL(18,2), processor_amount DECIMAL(18,2),
        difference DECIMAL(18,2), created_at TIMESTAMPTZ NOT NULL, source VARCHAR NOT NULL
    )
""")
conn.execute("""
    CREATE TABLE silver_enterprise_company (
        id BIGINT PRIMARY KEY, merchant_id VARCHAR NOT NULL,
        legal_name VARCHAR, trade_name VARCHAR, document VARCHAR,
        primary_cnae VARCHAR, created_at TIMESTAMPTZ, updated_at TIMESTAMPTZ
    )
""")

# ── Runs: two COMPLETED for 2024-01-08, one each for 2024-01-14 and 2024-01-15 ──
conn.execute("""
    INSERT INTO silver_reconciliation_runs VALUES
    (1,'2024-01-08','f1.csv','COMPLETED',3,'2024-01-08 08:00:00+00:00','2024-01-08 08:01:00+00:00','2024-01-08 08:00:00+00:00','test'),
    (2,'2024-01-08','f1.csv','COMPLETED',3,'2024-01-08 09:00:00+00:00','2024-01-08 09:01:00+00:00','2024-01-08 09:00:00+00:00','test'),
    (3,'2024-01-14','f2.csv','COMPLETED',2,'2024-01-14 08:00:00+00:00','2024-01-14 08:01:00+00:00','2024-01-14 08:00:00+00:00','test'),
    (4,'2024-01-15','f3.csv','COMPLETED',1,'2024-01-15 08:00:00+00:00','2024-01-15 08:01:00+00:00','2024-01-15 08:00:00+00:00','test')
""")

# ── Results for run 1 (superseded for 2024-01-08) ──
conn.execute("""
    INSERT INTO silver_reconciliation_results VALUES
    (10,1,'txn-A','M001','MATCHED',          100.00,100.00,0.00,'2024-01-08 08:01:00+00:00','test'),
    (11,1,'txn-B','M002','MISMATCHED',       200.00,195.00,5.00,'2024-01-08 08:01:00+00:00','test'),
    (12,1,'txn-C','M003','UNRECONCILED_INTERNAL',300.00,NULL,NULL,'2024-01-08 08:01:00+00:00','test')
""")

# ── Results for run 2 (winner for 2024-01-08) ──
conn.execute("""
    INSERT INTO silver_reconciliation_results VALUES
    (20,2,'txn-A','M001','MATCHED',          100.00,100.00,0.00,'2024-01-08 09:01:00+00:00','test'),
    (21,2,'txn-D','M001','MATCHED',           50.00, 50.00,0.00,'2024-01-08 09:01:00+00:00','test'),
    (22,2,'txn-C','M003','UNRECONCILED_INTERNAL',300.00,NULL,NULL,'2024-01-08 09:01:00+00:00','test')
""")

# ── Results for run 3 (2024-01-14, Sunday) ──
conn.execute("""
    INSERT INTO silver_reconciliation_results VALUES
    (30,3,'txn-E','M001','MATCHED',    150.00,150.00,0.00,'2024-01-14 08:01:00+00:00','test'),
    (31,3,'txn-F','M002','MISMATCHED', 250.00,240.00,10.0,'2024-01-14 08:01:00+00:00','test')
""")

# ── Results for run 4 (2024-01-15, Monday — next week) ──
conn.execute("""
    INSERT INTO silver_reconciliation_results VALUES
    (40,4,'txn-G','M001','MATCHED',120.00,120.00,0.00,'2024-01-15 08:01:00+00:00','test')
""")

# ── Merchant master data ──
conn.execute("""
    INSERT INTO silver_enterprise_company VALUES
    (1,'M001','ACME Corp Ltda',     'ACME','12.345.678/0001-90','6201-5','2024-01-01 00:00:00+00:00','2024-01-01 00:00:00+00:00'),
    (2,'M002','Beta Comercio SA',   'Beta','98.765.432/0001-10','4711-3','2024-01-01 00:00:00+00:00','2024-01-01 00:00:00+00:00'),
    (3,'M003','Gama Servicos ME',   'Gama','11.111.111/0001-11','6209-1','2024-01-01 00:00:00+00:00','2024-01-01 00:00:00+00:00')
""")

print('Fixture loaded.')
conn.execute('SELECT COUNT(*) AS runs FROM silver_reconciliation_runs').df()

Fixture loaded.


,runs
0,4


## Build gold layer

Passes the in-memory connection to `build()` so no file is written.

In [3]:
from src.gold.build import build

result = build(conn=conn)
for name, status in result.items():
    print(f'  {status.upper()}  gold_{name}')

INFO  Built: gold_ops_reconciliation_daily
INFO  Built: gold_ops_reconciliation_trend
INFO  Built: gold_cfo_weekly_summary
INFO  Built: gold_cfo_weekly_merchant_ranking
INFO  Built: gold_compliance_ledger
INFO  Gold layer build complete (5 artifacts).


  OK  gold_ops_reconciliation_daily
  OK  gold_ops_reconciliation_trend
  OK  gold_cfo_weekly_summary
  OK  gold_cfo_weekly_merchant_ranking
  OK  gold_compliance_ledger


## Test 1 — Winning-run policy

`reference_date = 2024-01-08` has two COMPLETED runs:
- run 1 started at 08:00 (superseded)
- run 2 started at 09:00 **(winner)**

**Ops** and **CFO** must aggregate only run 2. **Compliance** must see both.

In [4]:
# Ops daily: only winner run_id should appear for 2024-01-08
ops_runs = conn.execute("""
    SELECT DISTINCT run_id
    FROM gold_ops_reconciliation_daily
    WHERE reference_date = '2024-01-08'
""").fetchall()

ops_run_ids = {r[0] for r in ops_runs}
assert ops_run_ids == {2}, f'FAIL: expected {{2}}, got {ops_run_ids}'
print('PASS  Ops daily shows only run 2 for 2024-01-08')

conn.execute("""
    SELECT reference_date, run_id, category, txn_count, pct_of_total
    FROM gold_ops_reconciliation_daily
    WHERE reference_date = '2024-01-08'
    ORDER BY category
""").df()

PASS  Ops daily shows only run 2 for 2024-01-08


,reference_date,run_id,category,txn_count,pct_of_total
0,2024-01-08,2,MATCHED,2,0.666667
1,2024-01-08,2,UNRECONCILED_INTERNAL,1,0.333333


In [5]:
# CFO weekly: MATCHED txn_count for week 2024-01-08 should be 3
# (2 from run 2 on 2024-01-08) + (1 from run 3 on 2024-01-14)
# If superseded run 1 leaked in, MATCHED count would be higher
matched = conn.execute("""
    SELECT txn_count
    FROM gold_cfo_weekly_summary
    WHERE week_start = '2024-01-08' AND category = 'MATCHED'
""").fetchone()[0]

assert matched == 3, f'FAIL: expected 3 MATCHED, got {matched}'
print(f'PASS  CFO weekly MATCHED txn_count = {matched} (run 1 correctly excluded)')

conn.execute("""
    SELECT week_start, week_end, category, txn_count, amount_brl
    FROM gold_cfo_weekly_summary
    ORDER BY week_start, category
""").df()

PASS  CFO weekly MATCHED txn_count = 3 (run 1 correctly excluded)


,week_start,week_end,category,txn_count,amount_brl
0,2024-01-08,2024-01-14,MATCHED,3,300.0
1,2024-01-08,2024-01-14,MISMATCHED,1,240.0
2,2024-01-08,2024-01-14,UNRECONCILED_INTERNAL,1,300.0
3,2024-01-15,2024-01-21,MATCHED,1,120.0


In [ ]:
# Compliance: superseded run 1 AND winner run 2 must both appear
compliance_runs = {
    r[0] for r in conn.execute("""
        SELECT DISTINCT run_id
        FROM gold_compliance_ledger
        WHERE reference_date = '2024-01-08'
    """).fetchall()
}
assert 1 in compliance_runs, 'FAIL: compliance missing superseded run 1'
assert 2 in compliance_runs, 'FAIL: compliance missing winner run 2'
print(f'PASS  Compliance sees run_ids {compliance_runs} for 2024-01-08 (both superseded + winner)')

conn.execute("""
    SELECT run_id, run_status, transaction_id, category, internal_amount, processor_amount
    FROM gold_compliance_ledger
    WHERE reference_date = '2024-01-08'
    ORDER BY run_id, transaction_id
""").df()

## Test 2 — CFO COALESCE convention

`UNRECONCILED_INTERNAL` rows have `processor_amount = NULL`.
`COALESCE(processor_amount, internal_amount)` must fall back to `internal_amount`.

Run 2 on 2024-01-08 has txn-C: `internal_amount = 300.00`, `processor_amount = NULL`.
Expected `amount_brl` for UNRECONCILED_INTERNAL = **300.00**.

In [ ]:
amount_brl = conn.execute("""
    SELECT amount_brl
    FROM gold_cfo_weekly_summary
    WHERE week_start = '2024-01-08' AND category = 'UNRECONCILED_INTERNAL'
""").fetchone()[0]

assert float(amount_brl) == 300.00, f'FAIL: expected 300.00, got {amount_brl}'
print(f'PASS  UNRECONCILED_INTERNAL amount_brl = {float(amount_brl):.2f} (internal_amount used, no processor_amount)')

## Test 3 — Closed-week bucketing (Mon–Sun)

The week definition is ISO: Monday opens, Sunday closes.

- 2024-01-14 is a **Sunday** → `week_start = 2024-01-08`
- 2024-01-15 is the next **Monday** → `week_start = 2024-01-15`

They must land in **different** weeks.

In [ ]:
weeks = {
    str(r[0])
    for r in conn.execute('SELECT DISTINCT week_start FROM gold_cfo_weekly_summary').fetchall()
}
assert '2024-01-08' in weeks, f'FAIL: week starting 2024-01-08 not found — got {weeks}'
assert '2024-01-15' in weeks, f'FAIL: week starting 2024-01-15 not found — got {weeks}'
assert len(weeks) == 2,       f'FAIL: expected exactly 2 weeks, got {weeks}'
print(f'PASS  Distinct week_starts = {sorted(weeks)}')
print('      2024-01-14 (Sun) bucketed into 2024-01-08 week')
print('      2024-01-15 (Mon) bucketed into 2024-01-15 week')

conn.execute("""
    SELECT week_start, week_end, COUNT(*) AS categories, SUM(txn_count) AS total_txns
    FROM gold_cfo_weekly_summary
    GROUP BY week_start, week_end
    ORDER BY week_start
""").df()

## Test 4 — Idempotency

Running `build()` a second time must produce identical row counts and data.
Views are always `CREATE OR REPLACE` (trivially idempotent).
Tables use `CREATE OR REPLACE TABLE` — the second run drops and recreates them from the same silver data.

In [ ]:
tables = ['cfo_weekly_summary', 'cfo_weekly_merchant_ranking']

counts_before = {t: conn.execute(f'SELECT COUNT(*) FROM gold_{t}').fetchone()[0] for t in tables}
build(conn=conn)  # second build
counts_after  = {t: conn.execute(f'SELECT COUNT(*) FROM gold_{t}').fetchone()[0] for t in tables}

assert counts_before == counts_after, f'FAIL: row counts changed — before={counts_before} after={counts_after}'
print('PASS  Row counts identical after second build:')
for t in tables:
    print(f'      gold_{t}: {counts_before[t]} rows')

## Test 5 — Compliance ledger shows superseded runs

Regression guard: someone might "helpfully" add a winning-run filter to the compliance view.
This test explicitly verifies that:
1. Superseded run 1 (3 rows) is visible
2. Total compliance row count equals total `silver_reconciliation_results` row count

In [ ]:
run1_count = conn.execute("""
    SELECT COUNT(*) FROM gold_compliance_ledger WHERE run_id = 1
""").fetchone()[0]
assert run1_count == 3, f'FAIL: expected 3 rows from superseded run 1, got {run1_count}'
print(f'PASS  Superseded run 1 contributes {run1_count} rows to compliance ledger')

gold_total  = conn.execute('SELECT COUNT(*) FROM gold_compliance_ledger').fetchone()[0]
silver_total = conn.execute('SELECT COUNT(*) FROM silver_reconciliation_results').fetchone()[0]
assert gold_total == silver_total, (
    f'FAIL: compliance ledger ({gold_total}) != silver results ({silver_total}) — '
    'winning-run filter may have been accidentally added'
)
print(f'PASS  Compliance total ({gold_total}) == silver_reconciliation_results total ({silver_total})')

## Summary

In [ ]:
print('All assertions passed.')
print()
print('Gold artifacts built:')
for name in ['ops_reconciliation_daily','ops_reconciliation_trend',
             'cfo_weekly_summary','cfo_weekly_merchant_ranking','compliance_ledger']:
    n = conn.execute(f'SELECT COUNT(*) FROM gold_{name}').fetchone()[0]
    print(f'  gold_{name}: {n} rows/records')

conn.close()